# 00 Colab Bootstrap

Hand-maintained revision runbook. Do not regenerate from `scripts/revision/create_revision_notebooks.py` unless that generator is first updated to the current revision protocol.


## Purpose

Use this first when opening Colab directly. It mounts Drive, checks large artifacts, clones or updates the GitHub branch used as the code transport layer, and prepares the Mamba/Torch runtime required by Notebook 02a.

In [ ]:
from pathlib import Path
import os
import subprocess

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or was stale: {exc}')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(
        f'Google Drive root is not readable at {DRIVE_ROOT}. Restart the Colab runtime '
        'or remount with force_remount=True before continuing; cache absence cannot be '
        'inferred from an unhealthy mount.'
    )
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'

print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)


## Verify Drive Artifacts

In [ ]:
required_files = [
    'WFDB-ChapmanShaoxing.zip',
    'PTB-XL.zip',
    'Georgia.zip',
    'cpsc2021.zip',
]

missing = []
for name in required_files:
    path = DRIVE_ROOT / name
    print(f'{name}: exists={path.exists()} path={path}')
    if not path.exists():
        missing.append(str(path))

model_pointer = DRIVE_ROOT / 'model_runs' / 'current_final_ema_model_dir.txt'
if model_pointer.is_file() and model_pointer.read_text(encoding='utf-8').strip():
    model_dir = Path(model_pointer.read_text(encoding='utf-8').strip()).expanduser()
else:
    model_dir = DRIVE_ROOT / 'model'
legacy_fold_ckpts = sorted(model_dir.glob('fold*_best.pt'))
explicit_fold_ckpts = sorted(model_dir.glob('fold*_final_ema.pt'))
print('\nmodel_dir       :', model_dir)
print('model pointer   :', model_pointer, 'exists=', model_pointer.is_file())
print('legacy best checkpoints     :', len(legacy_fold_ckpts))
print('explicit final EMA checkpoints:', len(explicit_fold_ckpts))
mirror_root = MIRROR_REVISION_ROOT
print('Drive mirror root:', mirror_root, 'exists=', mirror_root.exists())
print('fold PCA manifest in mirror:', (mirror_root / 'manifests/fold_pca_manifest.json').exists())
checkpoint_patterns = {
    'ResNet checkpoints': 'experimental/resnet1d_cnn_checkpoints/*.pt',
    'Raw Mamba checkpoints': 'experimental/raw_mamba_checkpoints/*.pt',
    'Transformer checkpoints': 'experimental/transformer_ecg_checkpoints/*.pt',
    'Frozen-transform MLP checkpoints': 'experimental/hybrid_morphology_checkpoints/*.pt',
    'Fold prediction caches': 'predictions/folds/*.npz',
    'Comparator stress predictions': 'predictions/robustness_*_predictions.npz',
    'Command logs': 'logs/*.log',
}
for label, pattern in checkpoint_patterns.items():
    files = sorted(mirror_root.glob(pattern)) if mirror_root.exists() else []
    total_mb = sum(path.stat().st_size for path in files) / (1024 ** 2)
    print(f'{label:36s}: count={len(files):3d} size={total_mb:9.1f} MiB')

if len(legacy_fold_ckpts) < 5 and len(explicit_fold_ckpts) < 5:
    print('Checkpoint note: no complete five-fold set is present yet. This is expected before Notebook 02a retraining.')
if missing:
    raise FileNotFoundError('Missing required Drive artifacts:\n' + '\n'.join(missing))


## Clone Or Update Repo

In [ ]:
def run(cmd, cwd=None, log_path=None):
    print(f'$ {cmd}', flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        return subprocess.run(cmd, shell=True, cwd=str(run_cwd), check=True)
    log_path = Path(log_path)
    if not log_path.is_absolute():
        log_path = run_cwd / log_path
    log_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)
    from contextlib import ExitStack
    with ExitStack() as stack:
        log = stack.enter_context(log_path.open('w', encoding='utf-8'))
        durable_log = stack.enter_context(durable_log_path.open('w', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd, shell=True, cwd=str(run_cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            durable_log.write(line)
            durable_log.flush()
        return_code = proc.wait()
    print('Command log:', log_path)
    print('Durable command log:', durable_log_path)
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)

if (REPO_DIR / '.git').exists():
    run('git fetch origin', cwd=REPO_DIR)
    run(f'git checkout {BRANCH}', cwd=REPO_DIR)
    run(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repo. Rename it or set REPO_DIR to another folder.')
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
run('git rev-parse --short HEAD')
run('git status --short --branch')

required_reviewer_sources = [
    'scripts/revision/13_final_evidence_matrix.py',
    'scripts/revision/28_claim_readiness_gates.py',
    'scripts/revision/29_reviewer_presentation_assets.py',
    'scripts/revision/30_pooling_sensitivity_external.py',
    'scripts/revision/31_generate_external_comparator_predictions.py',
    'scripts/revision/32_paired_external_comparators.py',
    'scripts/revision/33_group_safe_score_calibration.py',
    'scripts/revision/34_extract_external_representations.py',
    'scripts/revision/35_true_fewshot_head_adaptation.py',
    'scripts/revision/36_build_marked_manuscript.py',
    'scripts/revision/37_artifact_source_audit.py',
    'scripts/revision/38_pipeline_storage_audit.py',
    'docs/revision_plan/georgia_label_mapping_review_20260703.csv',
]
missing_reviewer_sources = [relative for relative in required_reviewer_sources if not (REPO_DIR / relative).is_file()]
if missing_reviewer_sources:
    raise RuntimeError(
        'The checked-out branch is stale for the direct-run reviewer pipeline. Missing files: '
        + ', '.join(missing_reviewer_sources)
        + '. Commit/push the current notebook and revision-runner changes, then rerun Notebook 00.'
    )
required_source_tokens = {
    'scripts/revision/13_final_evidence_matrix.py': ['pre_specified_0.10_no_test_set_budget_selection', 'model_filter="full"'],
    'scripts/revision/28_claim_readiness_gates.py': ['adaptation_grid_issues', 'manifest_output_issues', 'primary_endpoint_issues', 'pre_specified_before_test_metric_evaluation'],
    'scripts/revision/artifact_mirror.py': ['merge_verified_no_prune', 'preserved_existing_count', '--verify-existing', '--include-prefix', '--source-conflict-policy'],
    'scripts/revision/01_generate_predictions.py': ['--fold-cache-dir'],
    'scripts/revision/03_generate_external_predictions.py': ['validate_checkpoint_files_against_oof_run_manifest'],
    'scripts/revision/12_robustness_stress.py': ['prediction_contract_hash', 'save_npz_compressed_atomic'],
    'scripts/revision/14_resnet1d_cnn_baseline.py': ['--fold-cache-dir', '--reuse-checkpoints'],
    'scripts/revision/16_raw_mamba_baseline.py': ['--fold-cache-dir', '--reuse-checkpoints'],
    'scripts/revision/23_generate_comparator_stress_predictions.py': ['--finalize-manifest-only', 'canonical_contract'],
    'scripts/revision/21_robustness_multicomparator.py': ['robustness_minirocket_clean_ref_predictions.npz', 'validate_stress_provenance'],
    'scripts/revision/26_hybrid_morphology_baseline.py': ['--fold-cache-dir', '--reuse-checkpoints'],
    'scripts/revision/29_reviewer_presentation_assets.py': ['reviewer_completion_input_contract.json'],
    'scripts/revision/31_generate_external_comparator_predictions.py': ['validate_checkpoint_set'],
    'scripts/revision/32_paired_external_comparators.py': ['probabilities must be in [0,1]'],
    'scripts/revision/33_group_safe_score_calibration.py': ['--primary-fraction', 'pre_specified_before_test_metric_evaluation', 'independent_target_groups_from_adaptation_pool'],
    'scripts/revision/34_extract_external_representations.py': ['checkpoint_source_contract', 'frozen_encoder_external_record_representation_v2_source_bound', 'validate_source_provenance', 'current runner/canonical contract'],
    'scripts/revision/35_true_fewshot_head_adaptation.py': ['frozen_encoder_true_linear_head_adaptation_v2_group_safe_gated', 'Embedding manifest is stale or incomplete', 'pre_specified_before_test_metric_evaluation', 'primary_endpoint_rows'],
    'scripts/revision/37_artifact_source_audit.py': ['canonical_is_authoritative', 'apply-legacy-only'],
    'scripts/revision/38_pipeline_storage_audit.py': ['complete_manifested', 'logs_are_durable_but_not_manifested', 'learned_prediction_checkpoint_contracts', 'fold_pca_models', 'recomputable_feature_caches', 'resolve_declared_path', 'true_fewshot_metric_cache'],
}
stale_sources = []
for relative, tokens in required_source_tokens.items():
    source_text = (REPO_DIR / relative).read_text(encoding='utf-8')
    missing_tokens = [token for token in tokens if token not in source_text]
    if missing_tokens:
        stale_sources.append(f'{relative}: {missing_tokens}')
if stale_sources:
    raise RuntimeError('Reviewer-pipeline source files are stale: ' + '; '.join(stale_sources))
print('Reviewer-pipeline source preflight: OK')


## Canonical Artifact Source Audit

The canonical mirror is the only operational artifact source. The Drive checkout is audited as legacy data and is never used automatically for restore.


In [ ]:
# One source of truth for revision artifacts.
import json
CANONICAL_REVISION_MIRROR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LEGACY_DRIVE_REVISION = DRIVE_ROOT / 'ECG-RAMBA' / 'reports' / 'revision'
MIGRATE_LEGACY_ONLY_ARTIFACTS = False

source_audit_command = (
    'python -u scripts/revision/37_artifact_source_audit.py '
    f'--canonical-root "{CANONICAL_REVISION_MIRROR}" '
    f'--legacy-root "{LEGACY_DRIVE_REVISION}" --reuse-existing'
)
if MIGRATE_LEGACY_ONLY_ARTIFACTS:
    # Canonical files always win. Only paths absent from canonical are imported.
    source_audit_command = source_audit_command.replace(' --reuse-existing', '')
    source_audit_command += ' --apply-legacy-only --keep-canonical-on-conflict'

run(
    source_audit_command,
    cwd=REPO_DIR,
    log_path='reports/revision/logs/artifact_source_audit.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{CANONICAL_REVISION_MIRROR}"',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/artifact_source_audit_mirror_publish.log',
)

source_audit_json = REPO_DIR / 'reports/revision/manifests/artifact_source_audit.json'
source_audit = json.loads(source_audit_json.read_text(encoding='utf-8'))
print('Canonical artifact root:', CANONICAL_REVISION_MIRROR)
print('Legacy audit-only root :', LEGACY_DRIVE_REVISION, 'exists=', LEGACY_DRIVE_REVISION.exists())
print('Artifact source counts :', source_audit.get('counts', {}))
print('Migration requested    :', source_audit.get('migration_requested'))
print('Migrated legacy-only   :', source_audit.get('migrated_legacy_only_count'))

conflicts = [row for row in source_audit.get('rows', []) if row.get('status') == 'conflict']
legacy_only = [row for row in source_audit.get('rows', []) if row.get('status') == 'legacy_only']
if conflicts:
    print(f'WARNING: {len(conflicts)} legacy files conflict with canonical files. Canonical versions remain authoritative.')
    for row in conflicts[:20]:
        print('  conflict:', row['relative_path'])
if legacy_only and not MIGRATE_LEGACY_ONLY_ARTIFACTS:
    print(f'NOTICE: {len(legacy_only)} files exist only in the legacy Drive checkout.')
    print('Review table_artifact_source_audit.csv, then set MIGRATE_LEGACY_ONLY_ARTIFACTS=True once if they are needed.')
else:
    print('No unapplied legacy-only artifact migration is required.')


## Install Lightweight Dependencies

In [ ]:
!python --version
!pip install -q "numpy>=2.0,<2.1" "scipy>=1.14.1,<2.0" "pandas==2.2.2" "scikit-learn>=1.2,<1.9" "requests==2.32.4" "pillow>=8,<12" tqdm "wfdb==4.1.2" joblib matplotlib seaborn packaging neurokit2 iterative-stratification thop


## Canonical Pipeline Storage Audit

This audit reports whether every fold cache, learned-comparator checkpoint, and stress-prediction family is complete and registered in the canonical mirror manifest. It is intentionally non-strict during bootstrap because unfinished reviewer experiments are valid intermediate states.

In [ ]:
storage_audit_command = (
    'python -u scripts/revision/38_pipeline_storage_audit.py '
    f'--canonical-root "{CANONICAL_REVISION_MIRROR}"'
)
run(
    storage_audit_command,
    cwd=REPO_DIR,
    log_path='reports/revision/logs/pipeline_storage_audit.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{CANONICAL_REVISION_MIRROR}"',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/pipeline_storage_audit_mirror_publish.log',
)
storage_audit_json = CANONICAL_REVISION_MIRROR / 'metrics' / 'pipeline_storage_audit.json'
storage_audit_table = CANONICAL_REVISION_MIRROR / 'tables' / 'table_pipeline_storage_audit.csv'
if not storage_audit_json.exists() or storage_audit_json.stat().st_size == 0:
    raise FileNotFoundError(f'Canonical storage audit output is missing: {storage_audit_json}')
storage_audit = json.loads(storage_audit_json.read_text(encoding='utf-8'))
print('Canonical storage audit status:', storage_audit.get('status'))
print('Durable logs on Drive       :', storage_audit.get('durable_log_count'))
print('Incomplete reviewer stages :', storage_audit.get('incomplete_required_stages'))
print('Invalid manifest contracts :', storage_audit.get('invalid_manifest_files'))
print('Storage audit table         :', storage_audit_table)
for row in storage_audit.get('stages', []):
    print(
        f"{row['stage']:<34} {row['status']:<24} "
        f"found={row['found_count']}/{row['expected_count']} "
        f"manifested={row['manifest_covered_count']}"
    )


## Verify GPU Runtime

In [ ]:
import sys

try:
    import torch
except Exception as exc:
    raise RuntimeError('PyTorch is not importable after base dependency setup. Restart the runtime and rerun Notebook 00.') from exc

print('Python:', sys.version.split()[0])
print('Torch :', torch.__version__, 'CUDA:', torch.version.cuda or 'CPU', 'available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'This Colab runtime is CPU-only, but ECG-RAMBA Mamba training/inference requires CUDA-enabled PyTorch. '
        'In Colab select Runtime > Change runtime type > Hardware accelerator > GPU, preferably A100 High-RAM; '
        'then disconnect/delete this CPU runtime and rerun Notebook 00 from the first cell.'
    )


## Install Model Dependencies / Mamba Runtime

In [ ]:
import json
from pathlib import Path

installer_nb = json.loads((REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb').read_text(encoding='utf-8'))
installer_source = None
for cell in installer_nb['cells']:
    if cell.get('cell_type') != 'code':
        continue
    source = ''.join(cell.get('source', []))
    if 'INSTALL_MODEL_DEPS = True' in source and 'AUTO_PIN_TORCH_FOR_MAMBA' in source:
        installer_source = source
        break
if installer_source is None:
    raise RuntimeError('Could not locate the canonical Mamba installer cell in notebook 02.')
print('Running canonical Mamba installer from Notebook 02...')
exec(compile(installer_source, str(REPO_DIR / 'notebooks/02_predictions_and_external_eval.ipynb:cell7'), 'exec'), globals(), globals())


## Run A0 Audit

In [ ]:
os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.chdir(REPO_DIR)
run(
    'python -u scripts/revision/00_audit_protocol.py',
    cwd=REPO_DIR,
    log_path='reports/revision/logs/bootstrap_protocol_audit.log',
)
